# Retrieval Quickstart

This notebook walks you through the core ZeroEntropy workflow: create a collection, add some documents, and run search queries against them. We'll try all three retrieval modes (documents, snippets, and pages) so you can see how they differ.

### Prerequisites
- Python 3.12+
- `zeroentropy` SDK (`pip install zeroentropy`)
- A ZeroEntropy API key ([get one here](https://dashboard.zeroentropy.dev))

### What we'll cover
- Creating a collection
- Adding Markdown files as documents
- Querying with `top_documents`, `top_snippets`, and `top_pages`
- The difference between coarse and precise snippets

### Files in this directory

```
retrieval_quickstart/
├── retrieval_quickstart.ipynb   # this notebook
├── retrieval_search_example.py  # standalone script (uses a HuggingFace dataset)
├── sample_docs/                 # sample markdown files used below
│   ├── intro.md
│   ├── tutorial.md
│   └── api_reference.md
├── pyproject.toml
└── README.md
```

### Install dependencies

In [1]:
!uv pip install zeroentropy python-dotenv

Audited 2 packages in 5ms


### Set up the client

Load your API key and create an async client. We use `AsyncZeroEntropy` here since notebooks support `await` natively.

In [2]:
from zeroentropy import AsyncZeroEntropy, ConflictError
import os
import glob

api_key = os.getenv("ZEROENTROPY_API_KEY")
if not api_key:
    raise ValueError("API Key not found. Make sure your .env file has ZEROENTROPY_API_KEY.")

zclient = AsyncZeroEntropy(api_key=api_key)
print("Client initialized.")

Client initialized.


### Create a collection

Collections group your documents together. Think of them like folders or indexes.

In [3]:
COLLECTION_NAME = "embedding_demo"

try:
    await zclient.collections.add(collection_name=COLLECTION_NAME)
    print(f"Collection '{COLLECTION_NAME}' created.")
except ConflictError:
    print(f"Collection '{COLLECTION_NAME}' already exists.")

Collection 'embedding_demo' created.


### Add some documents

Let's read the Markdown files from `sample_docs/` and upload them to the collection.

In [4]:
sample_dir = "sample_docs"
md_files = sorted(glob.glob(os.path.join(sample_dir, "*.md")))

for filepath in md_files:
    filename = os.path.basename(filepath)
    with open(filepath, "r") as f:
        content = f.read()

    try:
        await zclient.documents.add(
            collection_name=COLLECTION_NAME,
            path=filename,
            content={"type": "text", "text": content},
        )
        print(f"Added: {filename} ({len(content)} chars)")
    except ConflictError:
        print(f"Already exists: {filename}")

print(f"\nDone. {len(md_files)} documents added to '{COLLECTION_NAME}'.")

Added: api_reference.md (2182 chars)
Added: intro.md (1470 chars)
Added: tutorial.md (1830 chars)

Done. 3 documents added to 'embedding_demo'.


### Wait for indexing

Documents aren't searchable right away. ZeroEntropy needs a few seconds to parse and index them. We'll just poll until it's done.

In [5]:
import asyncio

while True:
    status = await zclient.status.get_status(collection_name=COLLECTION_NAME)
    indexed = status.num_indexed_documents
    total = status.num_documents
    print(f"Indexed: {indexed}/{total}", end="\r")
    if indexed == total and total > 0:
        break
    await asyncio.sleep(2)

print(f"\nAll {total} documents indexed and ready for search.")

Indexed: 3/3
All 3 documents indexed and ready for search.


### Search with `top_documents`

`top_documents` returns whole documents ranked by relevance. Good for figuring out *which file* has the answer.

In [6]:
query = "How do I handle errors in my data pipeline?"

response = await zclient.queries.top_documents(
    collection_name=COLLECTION_NAME,
    query=query,
    k=3,
)

print(f"Query: \"{query}\"\n")
for i, result in enumerate(response.results, 1):
    print(f"  {i}. {result.path}  (score: {result.score:.4f})")

Query: "How do I handle errors in my data pipeline?"

  1. tutorial.md  (score: 1.8958)
  2. api_reference.md  (score: 1.6104)
  3. intro.md  (score: 1.3250)


### Search with `top_snippets` (coarse)

Snippets are chunks of text *within* a document. Coarse mode returns longer passages (~2000 chars) with more context around the match.

In [7]:
query = "How do I set up authentication with an API key?"

coarse = await zclient.queries.top_snippets(
    collection_name=COLLECTION_NAME,
    query=query,
    k=3,
    precise_responses=False,
)

print(f"Query: \"{query}\"")
print(f"Mode: COARSE snippets (~2000 chars)\n")
for i, result in enumerate(coarse.results, 1):
    snippet = result.content.replace("\n", " ")[:120] + "..." if result.content and len(result.content) > 120 else result.content
    print(f"  {i}. [{result.path}] score={result.score:.4f}")
    print(f"     {snippet}\n")

Query: "How do I set up authentication with an API key?"
Mode: COARSE snippets (~2000 chars)

  1. [api_reference.md] score=0.4259
     # API Reference  Complete reference for the DataFlow SDK. All classes and methods are available under the `dataflow` pac...

  2. [tutorial.md] score=0.2103
     # Quickstart Tutorial  This tutorial walks you through building your first DataFlow pipeline: a real-time word counter t...

  3. [intro.md] score=0.1663
     # Introduction to DataFlow  DataFlow is an open-source data pipeline framework designed for real-time stream processing ...



### Search with `top_snippets` (precise)

Precise mode returns shorter passages (~200 chars) that pinpoint the exact answer. Useful when you need tight, quotable text.

In [8]:
precise = await zclient.queries.top_snippets(
    collection_name=COLLECTION_NAME,
    query=query,
    k=3,
    precise_responses=True,
)

print(f"Query: \"{query}\"")
print(f"Mode: PRECISE snippets (~200 chars)\n")
for i, result in enumerate(precise.results, 1):
    snippet = result.content.replace("\n", " ") if result.content else ""
    print(f"  {i}. [{result.path}] score={result.score:.4f}")
    print(f"     {snippet}\n")

Query: "How do I set up authentication with an API key?"
Mode: PRECISE snippets (~200 chars)

  1. [api_reference.md] score=0.7130
     ---  ## Authentication  All API calls require a valid API key. Set the `DATAFLOW_API_KEY` environment variable or pass it directly:  ```python from dataflow import Pipeline, Config  config = Config(api_key="your-api-key-here")

  2. [api_reference.md] score=0.6039
     API keys can be generated from the DataFlow Dashboard under Settings > API Keys. Each key has configurable rate limits and permissions scoped to specific pipelines or namespaces.

  3. [api_reference.md] score=0.6025
     config = Config(api_key="your-api-key-here") pipeline = Pipeline("secure-pipeline", config=config) ```  API keys can be generated from the DataFlow Dashboard



### Search with `top_pages`

`top_pages` returns individual pages ranked by relevance. For plain text like our Markdown files each document counts as one page, but this really shines with multi-page PDFs.

In [9]:
query = "What are streams and transforms in a data pipeline?"

pages = await zclient.queries.top_pages(
    collection_name=COLLECTION_NAME,
    query=query,
    k=3,
    include_content=True,
)

print(f"Query: \"{query}\"\n")
for i, result in enumerate(pages.results, 1):
    preview = result.content.replace("\n", " ")[:100] + "..." if result.content and len(result.content) > 100 else result.content
    print(f"  {i}. {result.path} (page {result.page_index}, score: {result.score:.4f})")
    print(f"     {preview}\n")

Query: "What are streams and transforms in a data pipeline?"

  1. intro.md (page 0, score: 1.5144)
     # Introduction to DataFlow  DataFlow is an open-source data pipeline framework designed for real-tim...

  2. api_reference.md (page 0, score: 1.3706)
     # API Reference  Complete reference for the DataFlow SDK. All classes and methods are available unde...

  3. tutorial.md (page 0, score: 1.2269)
     # Quickstart Tutorial  This tutorial walks you through building your first DataFlow pipeline: a real...



### Clean up

Delete the collection when you're done experimenting.

In [10]:
await zclient.collections.delete(collection_name=COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' deleted.")

Collection 'embedding_demo' deleted.


### Done!

That's the full workflow: create a collection, add documents, wait for indexing, and search. You've seen all three query modes (`top_documents`, `top_snippets`, `top_pages`) and the difference between coarse and precise snippets.

If you want to try this on a bigger dataset, check out `retrieval_search_example.py` in this directory. It pulls a scientific abstracts corpus from HuggingFace and runs the same workflow.